# Lộc lớn — SAGE M³FD Fusion, 300 cặp

Chạy pretrained Student và đánh giá EN/SD/SCD/MS-SSIM. Xem `M3FD_PROTOCOL.md` trước khi chạy. Notebook gọi runner đã kiểm thử của repo; không huấn luyện segmentation.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/run_m3fd_baseline.py").is_file()), None)
assert ROOT is not None, "Mở notebook từ repo xulianh"
os.chdir(ROOT)
DATASET = ROOT / "data/raw/M3FD_Fusion"
UPSTREAM = ROOT / "external/SAGE_IVIF"
DEVICE = "cpu"  # đổi thành cuda nếu PyTorch CUDA có sẵn


## Kiểm tra dữ liệu
Phải đúng 300 ID, không thiếu ảnh, không trùng stem, kích thước IR/VIS khớp.


In [ ]:
subprocess.run([sys.executable, "scripts/run_m3fd_baseline.py", "--dataset-root", str(DATASET), "--audit-only"], check=True)


## Inference và evaluation
Cần cài `requirements-inference.txt` và checkout upstream đúng commit đã khóa. Runner tự tạo thư mục mới, giữ toàn bộ provenance; thời gian chạy tùy CPU/GPU.


In [ ]:
from datetime import datetime, timezone
RUN = ROOT / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_sage_m3fd_notebook")
subprocess.run([sys.executable, "scripts/run_m3fd_baseline.py", "--dataset-root", str(DATASET), "--upstream", str(UPSTREAM), "--device", DEVICE, "--output-dir", str(RUN)], check=True)


In [ ]:
config = json.loads((RUN / "run_config.json").read_text())
summary = json.loads((RUN / "metrics/summary.json").read_text())
assert config["status"] == "complete" and summary["valid_images"] == 300
for metric in ["EN", "SD", "SCD", "MS_SSIM"]:
    print(metric, summary["metrics"][metric])
from IPython.display import display, Image
display(Image(filename=str(RUN / "figures/preview_ir_vis_sage.png")))
